# 0. EDA & Data Preparation

Разведочный анализ транскриптомных когорт, определение таргетов и конфаундеров, baseline DR-визуализация эмбеддингов, подготовка train/test split.

**Пайплайн:** `0.eda_data_prep` → `1.batch_correction` → `2.metrics_modeling_stress_test`

**Вход:** `data/raw/PUB_*.adata.zarr/` (когорты в формате AnnData/Zarr)
**Выход:** `data/interim/prepared.adata.zarr` (объединённый датасет с Target_, Split_, survival metadata в .uns)

## 0. Imports & Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter
from loguru import logger
from sklearn.model_selection import StratifiedShuffleSplit

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    SEED,
    DATA_DIR,
    DATA_RAW_DIR,
    DATA_INTERIM_DIR,
    REPORTS_DIR,
    FIGURES_DIR,
    BATCH_COL,
    DIAGNOSIS_COL,
    SPLIT_PREFIX,
    TARGET_PREFIX,
)
from batchcor_rna_emb.data_io import load_all_cohorts, load_cohort, save_adata_zarr

# Loguru setup
set_logger(level="INFO")

# Reproducibility
np.random.seed(SEED)

# Plot params
mpl.rcParams["figure.dpi"] = 150
sns.set_style("ticks")

In [ ]:
# Директории для фигур
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

EDA_FIG_DIR = FIGURES_DIR / "eda"
EDA_FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Загрузка когорт

In [ ]:
# Загрузка всех когорт из data/raw/ (или data/ если когорты лежат в корне)
cohorts = load_all_cohorts(DATA_RAW_DIR)

for c in cohorts:
    batch_info = c.obs[BATCH_COL].value_counts().to_dict() if BATCH_COL in c.obs.columns else "N/A"
    diag_info = c.obs[DIAGNOSIS_COL].value_counts().to_dict() if DIAGNOSIS_COL in c.obs.columns else "N/A"
    logger.info("Когорта: {} | batch: {} | diagnosis: {}", c.shape, batch_info, diag_info)

In [ ]:
# Объединение когорт в один AnnData
dataset = ad.concat(cohorts, join="outer")
logger.info("Объединённый AnnData: {} samples x {} genes", dataset.n_obs, dataset.n_vars)
logger.info("Батч-распределение:\n{}", dataset.obs[BATCH_COL].value_counts())
logger.info("Диагноз-распределение:\n{}", dataset.obs[DIAGNOSIS_COL].value_counts())

# Доступные FM-эмбеддинги в .obsm
emb_keys = [k for k in dataset.obsm.keys() if k.startswith("FM_")]
logger.info("Доступные FM-эмбеддинги: {}", emb_keys)

# Выбор основного эмбеддинга для анализа
EMB_KEY = emb_keys[0] if emb_keys else None
logger.info("Рабочий эмбеддинг: '{}'", EMB_KEY)

## 2. Характеристика датасета

In [ ]:
# Общая статистика
logger.info("Число образцов: {}", dataset.n_obs)
logger.info("Число генов: {}", dataset.n_vars)
logger.info("Число батчей: {}", dataset.obs[BATCH_COL].nunique())
logger.info("Число диагнозов: {}", dataset.obs[DIAGNOSIS_COL].nunique())

# Crosstab: когорта × диагноз
ct = pd.crosstab(dataset.obs[BATCH_COL], dataset.obs[DIAGNOSIS_COL], margins=True)
display(ct)

In [ ]:
# Пропуски в .obs — показываем колонки с NaN
obs_na = dataset.obs.isna().sum()
obs_na_nonzero = obs_na[obs_na > 0].sort_values(ascending=False)
if len(obs_na_nonzero) > 0:
    logger.info("Колонки с пропусками:\n{}", obs_na_nonzero.to_string())
else:
    logger.info("Пропусков в .obs нет")

# Типы данных
display(dataset.obs.dtypes.value_counts())

## 3. Определение таргетов

### 3.1 Бинарная классификация (Responders vs Non-responders)

Адаптируйте `initial_target_name`, `filtered_target_name` и `category_mapper` под ваш датасет.

In [ ]:
# --- НАСТРОЙКИ ТАРГЕТА (адаптировать под данные) ---
initial_target_name = "Recist"  # исходная колонка с ответом на терапию
filtered_target_name = "Response_wo_SD"  # имя фильтрованного таргета

# Маппинг классов: Responders (1) vs Non-responders (0)
category_mapper = {"PD": 0, "CR": 1, "PR": 1}

# Формирование колонки Target_
target_col_name = f"{TARGET_PREFIX}{filtered_target_name}"
dataset.obs[target_col_name] = dataset.obs[initial_target_name].map(category_mapper).astype("float32")

# Статистика таргета
target_counts = dataset.obs[target_col_name].value_counts(dropna=False)
logger.info("Распределение таргета '{}':\n{}", target_col_name, target_counts)

# Обратный маппинг для визуализации
class_mapper = {0: "Non-responder", 1: "Responder"}
logger.info("class_mapper: {}", class_mapper)

### 3.2 Survival Data (опционально)

Определите тип выживаемости (OS/PFS) и соответствующие колонки.

In [ ]:
# --- НАСТРОЙКИ SURVIVAL (адаптировать под данные) ---
survival_type = "OS"  # или "PFS"
duration_col = survival_type  # колонка с временем (в днях)
event_col = f"{survival_type}_FLAG"  # колонка с событием (1=событие, 0=цензурирован)

has_survival = duration_col in dataset.obs.columns and event_col in dataset.obs.columns

if has_survival:
    # Конвертация в месяцы для визуализации
    durations_months = dataset.obs[duration_col] / 30.44

    kmf = KaplanMeierFitter()
    kmf.fit(
        durations=durations_months.dropna(),
        event_observed=dataset.obs.loc[durations_months.dropna().index, event_col],
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Распределение времени наблюдения
    axes[0].hist(durations_months.dropna(), bins=30, edgecolor="black", alpha=0.7)
    axes[0].set_xlabel("Время (месяцы)")
    axes[0].set_ylabel("Число пациентов")
    axes[0].set_title(f"{survival_type}: Распределение времени наблюдения")

    # KM-кривая (общая)
    kmf.plot_survival_function(ax=axes[1])
    axes[1].set_title(f"{survival_type}: Kaplan-Meier (все пациенты)")
    axes[1].set_xlabel("Время (месяцы)")

    plt.tight_layout()
    fig.savefig(EDA_FIG_DIR / f"{survival_type}_overview.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Сохраним параметры survival в .uns
    dataset.uns["survival_type"] = survival_type
    dataset.uns["survival_duration_col"] = duration_col
    dataset.uns["survival_event_col"] = event_col
    logger.info("Survival data сохранён в .uns: type={}, N={}", survival_type, durations_months.notna().sum())
else:
    logger.warning("Колонки survival ({}, {}) не найдены в .obs", duration_col, event_col)

## 4. Конфаундеры

Определите потенциальные конфаундеры — факторы, влияющие на исход и связанные с батч-эффектом (Stage, Age, Gender, Therapy и т.д.).

In [ ]:
# --- НАСТРОЙКИ КОНФАУНДЕРОВ (адаптировать под данные) ---
categorical_confounders: list[str] = []  # например: ["Gender", "Stage"]
numeric_confounders: list[str] = []  # например: ["Age", "TMB"]

# Показать доступные колонки для выбора конфаундеров
logger.info("Доступные колонки в .obs:\n{}", list(dataset.obs.columns))

In [ ]:
# Визуализация категориальных конфаундеров × таргет
for cat_col in categorical_confounders:
    if cat_col not in dataset.obs.columns:
        logger.warning("Колонка '{}' не найдена в .obs, пропускаем", cat_col)
        continue

    crosstab = pd.crosstab(
        dataset.obs[cat_col],
        dataset.obs[target_col_name].map(class_mapper).fillna("NaN"),
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Проценты
    crosstab_pct = crosstab.div(crosstab.sum(axis=1), axis=0)
    crosstab_pct.plot.bar(stacked=True, ax=axes[0], edgecolor="black")
    axes[0].set_title(f"{cat_col} × {filtered_target_name} (проценты)")
    axes[0].set_ylabel("Доля")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].legend(title="Target")

    # Абсолютные значения
    crosstab.plot.bar(stacked=True, ax=axes[1], edgecolor="black")
    axes[1].set_title(f"{cat_col} × {filtered_target_name} (абсолютные)")
    axes[1].set_ylabel("Число образцов")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].legend(title="Target")

    plt.tight_layout()
    fig.savefig(EDA_FIG_DIR / f"confounder_{cat_col}_barplot.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# Визуализация числовых конфаундеров × таргет
for num_col in numeric_confounders:
    if num_col not in dataset.obs.columns:
        logger.warning("Колонка '{}' не найдена в .obs, пропускаем", num_col)
        continue

    target_series = dataset.obs[target_col_name].map(class_mapper).dropna()
    num_series = dataset.obs.loc[target_series.index, num_col].dropna()
    common_idx = target_series.index.intersection(num_series.index)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Гистограмма
    for label in sorted(target_series.loc[common_idx].unique()):
        mask = target_series.loc[common_idx] == label
        axes[0].hist(num_series.loc[common_idx][mask], bins=20, alpha=0.6, label=label, density=True)
    axes[0].set_title(f"{num_col}: распределение по {filtered_target_name}")
    axes[0].set_xlabel(num_col)
    axes[0].legend()

    # Boxplot
    box_data = pd.DataFrame({num_col: num_series.loc[common_idx], "Target": target_series.loc[common_idx]})
    sns.boxplot(data=box_data, x="Target", y=num_col, ax=axes[1])
    axes[1].set_title(f"{num_col} by {filtered_target_name}")

    plt.tight_layout()
    fig.savefig(EDA_FIG_DIR / f"confounder_{num_col}_boxplot.png", dpi=300, bbox_inches="tight")
    plt.show()

## 5. Baseline DR Analysis (до батч-коррекции)

PCA и UMAP на raw (некорректированных) FM-эмбеддингах, окрашенные по batch и diagnosis.

In [ ]:
# UMAP на raw эмбеддингах для каждой FM-модели
UMAP_DIST = 0.4
UMAP_SPREAD = 1.0

for emb_key in emb_keys:
    logger.info("DR Analysis для эмбеддинга: '{}'", emb_key)

    # kNN граф + UMAP
    sc.pp.neighbors(dataset, use_rep=emb_key, n_neighbors=30, metric="cosine")
    sc.tl.umap(dataset, min_dist=UMAP_DIST, spread=UMAP_SPREAD)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # По batch
    sc.pl.umap(dataset, color=BATCH_COL, s=5, title=f"{emb_key} — Batch", ax=axes[0], show=False)
    # По diagnosis
    sc.pl.umap(dataset, color=DIAGNOSIS_COL, s=5, title=f"{emb_key} — Diagnosis", ax=axes[1], show=False)

    plt.tight_layout()
    safe_name = emb_key.replace("/", "_")
    fig.savefig(EDA_FIG_DIR / f"baseline_umap_{safe_name}.png", dpi=300, bbox_inches="tight")
    plt.show()

## 6. Train/Test Split

Стратификация по таргету + batch (на уровне пациентов, не образцов — избежать утечки данных).

In [ ]:
# Стратифицированный split по target + batch
split_col_name = f"{SPLIT_PREFIX}{filtered_target_name}"
test_size = 0.2

# Отбираем только образцы с валидным таргетом
annotated_mask = dataset.obs[target_col_name].notna()
annotated_obs = dataset.obs.loc[annotated_mask]

# Комбинированная стратификация: target + batch
strat_label = (
    annotated_obs[target_col_name].astype(str) + "_" + annotated_obs[BATCH_COL].astype(str)
)

# Если какой-то strat_label имеет < 2 образцов, fallback на стратификацию только по target
label_counts = strat_label.value_counts()
if (label_counts < 2).any():
    logger.warning(
        "Некоторые strat-группы ({}) имеют < 2 образцов, стратификация только по target",
        (label_counts < 2).sum(),
    )
    strat_label = annotated_obs[target_col_name].astype(str)

sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
train_idx, test_idx = next(sss.split(annotated_obs, strat_label))

train_samples = annotated_obs.index[train_idx]
test_samples = annotated_obs.index[test_idx]

# Записываем split в .obs
dataset.obs[split_col_name] = np.nan
dataset.obs.loc[train_samples, split_col_name] = "train"
dataset.obs.loc[test_samples, split_col_name] = "test"

logger.info(
    "Train/Test split '{}': train={}, test={}, unannotated={}",
    split_col_name,
    len(train_samples),
    len(test_samples),
    (~annotated_mask).sum(),
)

In [ ]:
# Визуализация распределения классов по сплитам
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (split_name, split_df) in zip(
    axes, dataset.obs.groupby(split_col_name, observed=True)
):
    counts = split_df[target_col_name].map(class_mapper).value_counts()
    counts.plot.bar(ax=ax, edgecolor="black", color=["#FB6E5B", "#7BE76C"])
    ax.set_title(f"{split_name.title()} (N={len(split_df)})")
    ax.set_ylabel("Число образцов")
    ax.tick_params(axis="x", rotation=0)

    # Проценты
    for i, (label, count) in enumerate(counts.items()):
        ax.text(i, count + 0.5, f"{count / len(split_df):.1%}", ha="center", fontsize=10)

fig.suptitle(f"Распределение {filtered_target_name} по сплитам", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(EDA_FIG_DIR / f"split_{filtered_target_name}_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# Распределение батчей по сплитам
ct_split_batch = pd.crosstab(
    dataset.obs[split_col_name].fillna("unannotated"),
    dataset.obs[BATCH_COL],
)
display(ct_split_batch)

## 7. Сохранение подготовленного датасета

Сохраняем объединённый AnnData с Target_, Split_ и survival metadata в `data/interim/prepared.adata.zarr` для дальнейшей батч-коррекции (notebook 1).

In [ ]:
# Метаданные в .uns для передачи между ноутбуками
dataset.uns["target_col"] = target_col_name
dataset.uns["split_col"] = split_col_name
dataset.uns["filtered_target_name"] = filtered_target_name
dataset.uns["category_mapper"] = category_mapper
dataset.uns["class_mapper"] = class_mapper
dataset.uns["embedding_keys"] = emb_keys

# Сохранение
output_path = DATA_INTERIM_DIR / "prepared.adata.zarr"
save_adata_zarr(dataset, output_path)

logger.info("Сохранён prepared AnnData: {}", output_path)
logger.info("  .obs колонки: {}", list(dataset.obs.columns))
logger.info("  .obsm ключи: {}", list(dataset.obsm.keys()))
logger.info("  .uns ключи: {}", list(dataset.uns.keys()))